# Creating Manifests and Archives

Two key features of the ProteinGym dataset are the manifest-based configuration and archiving of datasets. This notebook will demonstrate how to create manifest files, understand their structure, and create dataset archives. We'll use the NEIME 2019 dataset in `example_data/NEIME_2019` as our example. 

## What is a Manifest?

A manifest is a TOML configuration file that describes your dataset's structure and metadata. It serves as a blueprint for loading and organizing your protein data. TOML stands for Tom's Obvious Minimal Language and is designed as a configuration file that is easy to read and write. In this tutorial we will write the TOML file using the notebook cells, but you can also open the TOML in your favorite text editor.

## Manifest Structure

Let's examine the key sections of a manifest file:

In [1]:
# First, let's look at the example manifest
from pathlib import Path
import toml

# Read the example manifest
manifest_path = Path("../example_data/neime_2019.toml")
with open(manifest_path, 'r') as f:
    manifest_content = f.read()
    
print(manifest_content)

name = "NEIME_2019"
description = "The NEIME Kennouche 2019 (UniProt id: A0A1I9GEU1) datase"
version = "1.0.0"

[[ assay_conditions ]]
name = "PH"
description = "pH level of the samples"
unit = "pH"
value = 7

[[ assay_conditions ]]
name = "T"
description = "Temperature level of the samples"
unit = "C"
value = 37

[[ assays ]]
sequence = "mutated_sequence"
target = "DMS_score"
path = "./NEIME_2019/Assays/Assay1.csv"

[[ sequences ]]
sequence_type = "wild_type"
sequence_alphabet = "DNA"
path = "./NEIME_2019/sequences/A0A1I9GEU1.fasta"

[[ structures ]]
path = "./NEIME_2019/Structures/computational.pdb"
name = "A0A1I9GEU1"
description = "ALPHAFOLD MONOMER V2.0 PREDICTION FOR PILIN"

[structures.metadata]
source = "alphafold"
version = "v2.0"
avg_plddt = "90"

[[ msas ]]
path = "../example_data/NEIME_2019/MSA/msa.a2m"
format = "fasta"


## Creating a Complete Manifest

Let's create a complete manifest file step by step. A proper manifest should include:

### 1. Top-level Metadata

We give a version, name and description to the dataset. 

We follow [semantic versioning](https://semver.org/) to denote the version number. 
This follows the MAJOR.MINOR.PATCH increments. 
We recommend to update the major when the changes in the dataset would require a re-run of the benchmarks on the dataset, minor for adding extra information and patch for repairing small mistakes. 

For the name we recommend the current ProteinGym format of `<Uniprot_ID>_<SPECIES>_<Author>_<Year>`. 

The description field we leave up to the author to add any information to the dataset that is not captured by the standard fields.


In [2]:
# Create a complete manifest for the NEIME 2019 dataset
complete_manifest = {
    "version": "1.0.0",
    "name": "NEIME_2019",
    "description": "NEIME enzyme dataset from Kennouche et al. 2019 with DMS scores"
}

for key, value in complete_manifest.items():
    print(f"{key} = \"{value}\"")

version = "1.0.0"
name = "NEIME_2019"
description = "NEIME enzyme dataset from Kennouche et al. 2019 with DMS scores"


### 2. Assay Conditions

Here we define the assay_conditions that remain constant across the assay measurements. In this example we add two entries, one for temperature and one for the pH

In [3]:
# Define assay conditions
assay_conditions = [
    {
        "name": "temperature",
        "description": "Reaction temperature",
        "unit": "°C",
        "value": 37
    },
    {
        "name": "pH",
        "description": "Buffer pH",
        "unit": "pH",
        "value": 7.4
    }
]

complete_manifest["assay_conditions"] = assay_conditions

for condition in assay_conditions:
    print("[[ assay_conditions ]]")
    for key, value in condition.items():
        print(f"{key} = {value}")

[[ assay_conditions ]]
name = temperature
description = Reaction temperature
unit = °C
value = 37
[[ assay_conditions ]]
name = pH
description = Buffer pH
unit = pH
value = 7.4


### 3. Assays

Each Assay can consist of multiple Assay**s**. Here we highlight the entry for the example DMS assay. Currently each dataset in ProteinGym consists of either a DMS or ClinVar assay, but for the case of protein engineering it can be beneficial to record multiple assays.

Each Assay requireds three fields:
- **sequence**: targets the column in your assay csv that contains the sequence.
- **target**: targets the column with measurement values.
- **path**: path to the location of your assay csv.

In [4]:
# Define Assays
assays = [
    {"sequence" : "mutated_sequence",
     "target" : "DMS_score",
     "path" : "./NEIME_2019/Assays/Assay1.csv",
    }
]
complete_manifest["assays"] = assays

for assay in assays:
    print("[[ assays ]]")
    for key, value in assay.items():
        print(f"{key} = {value}")

[[ assays ]]
sequence = mutated_sequence
target = DMS_score
path = ./NEIME_2019/Assays/Assay1.csv


### 4. Sequences

We record the mutated sequences in sequences in the assay, but for some cases you might want to access the original sequence. For example, in the case of a constrastive learning mechanism you might want to contrast the mutated sequence to the wild-type sequence. For this you can add a sequence field to the dataset.

Each sequence section requires:
- **path**: path to the fasta or fastq file.
- **sequence_alphabet**: AA, DNA or RNA
- **sequence_type**: Indicate the type of the sequence, e.g. wild-type, engineered or a custom type

In [5]:
# Define sequences
sequences = [
    {
        "path": "NEIME_2019/sequences/A0A1I9GEU1.fasta",
        "sequence_alphabet": "AA",
        "sequence_type": "wild_type"
    }
]

complete_manifest["sequences"] = sequences

for seq in sequences:
    print("[[ sequences ]]")
    for key, value in seq.items():
        print(f"{key} = {value}")

[[ sequences ]]
path = NEIME_2019/sequences/A0A1I9GEU1.fasta
sequence_alphabet = AA
sequence_type = wild_type


### 5. Structures

You can add structures to the dataset as either a directory or single files. We allow for the possibility of loading PDBs, Cifs, and binary Cifs. Since meta data on the structure is usually included in the structure file itself, we recommend to only include the meta data for datasets with single structures. If you are loading a directory of structures you can access the structure meta data through the structure object. We'll show you how to access the meta data in `03_Loading_and_Accessing_Data.ipynb`

The following four entries are allowed for structures:
- **path**: Path to the structure file or directory
- **name**: Name of the protein, e.g. PDB ID or Uniprot ID
- **description**: Description of the structure
- **metadata**: Metadata fields

You can add metadata as a dictionary based entry, or add the metadata as a separete section. See the examples below:

In [6]:
# Define structures
structures = [
    {"path": "NEIME_2019/Structures/computational.pdb",
     "name": "A0A1I9GEU1",
     "description": "ALPHAFOLD MONOMER V2.0 PREDICTION FOR PILIN",
     "metadata": {"source" : "alphafold", "version" : "v2.0", "avg_plddt" : "90", "custom_field" : "custom_data" }}
]

# Alternatively you can add a new section with [structures.metadata] to the toml:
# [structures.metadata]
# source = "alphafold"
# version = "v2.0"
# avg_plddt = "90"

complete_manifest["structures"] = structures

for struct in structures:
    print("[[ structures ]]")
    for key, value in struct.items():
        print(f"{key} = {value}")

[[ structures ]]
path = NEIME_2019/Structures/computational.pdb
name = A0A1I9GEU1
description = ALPHAFOLD MONOMER V2.0 PREDICTION FOR PILIN
metadata = {'source': 'alphafold', 'version': 'v2.0', 'avg_plddt': '90', 'custom_field': 'custom_data'}


### 6. Multiple Sequence Alignments (MSAs)

We allow for the possibility to add a single MSA to the dataset, representing the evolutionary alignments of the protein of interest. Here we are limited to file formats [supported](https://biopython.org/wiki/AlignIO) by BioPython.

The following entries are allowed for msas:
- **path (required)**: Path to the file
- **name**: The name of the MSA
- **description**: The description of the MSA.
- **format (required)**: The format of the MSA file
- **metadata**: The fields for metadata

In [7]:
# Define MSAs
msas = [
    {
        "path": "NEIME_2019/MSA/msa.a2m",
        "format": "fasta",
        "name" : "A0A1I9GEU1_NEIME",
        "description" : "Generated by Software X",
        "metadata" : {"Software" : "MMSeqs2", "Setting1" : "30%"}
    }
]

complete_manifest["msas"] = msas

for msa in msas:
    print("[[ msas ]]")
    for key, value in msa.items():
        print(f"{key} = {value}")

[[ msas ]]
path = NEIME_2019/MSA/msa.a2m
format = fasta
name = A0A1I9GEU1_NEIME
description = Generated by Software X
metadata = {'Software': 'MMSeqs2', 'Setting1': '30%'}


## Writing the Manifest File

Now let's write our complete manifest to a TOML file:

In [8]:
# Write the complete manifest
output_path = Path("../example_data/neime_2019_complete.toml")

with open(output_path, 'w') as f:
    toml.dump(complete_manifest, f)


print(f"Complete manifest written to: {output_path}")

# Display the written content
with open(output_path, 'r') as f:
    print("\nGenerated manifest:")
    print(f.read())

Complete manifest written to: ../example_data/neime_2019_complete.toml

Generated manifest:
version = "1.0.0"
name = "NEIME_2019"
description = "NEIME enzyme dataset from Kennouche et al. 2019 with DMS scores"
[[assay_conditions]]
name = "temperature"
description = "Reaction temperature"
unit = "°C"
value = 37

[[assay_conditions]]
name = "pH"
description = "Buffer pH"
unit = "pH"
value = 7.4

[[assays]]
sequence = "mutated_sequence"
target = "DMS_score"
path = "./NEIME_2019/Assays/Assay1.csv"

[[sequences]]
path = "NEIME_2019/sequences/A0A1I9GEU1.fasta"
sequence_alphabet = "AA"
sequence_type = "wild_type"

[[structures]]
path = "NEIME_2019/Structures/computational.pdb"
name = "A0A1I9GEU1"
description = "ALPHAFOLD MONOMER V2.0 PREDICTION FOR PILIN"

[structures.metadata]
source = "alphafold"
version = "v2.0"
avg_plddt = "90"
custom_field = "custom_data"
[[msas]]
path = "NEIME_2019/MSA/msa.a2m"
format = "fasta"
name = "A0A1I9GEU1_NEIME"
description = "Generated by Software X"

[msas.met

## Loading and Validating the Manifest

Let's load our manifest and validate it:

In [9]:
from pg2_dataset import Manifest, Dataset

# Load the manifest
manifest = Manifest.from_path(output_path)

print(f"Loaded manifest: {manifest.name}")
print(f"Description: {manifest.description}")
print(f"Version: {manifest.version}")
print(f"Number of assays: {len(manifest.assays)}")
print(f"Number of sequences: {len(manifest.sequences)}")
print(f"Number of structures: {len(manifest.structures)}")
print(f"Number of MSAs: {len(manifest.msas)}")
print(f"Number of assay conditions: {len(manifest.assay_conditions)}")

Loaded manifest: NEIME_2019
Description: NEIME enzyme dataset from Kennouche et al. 2019 with DMS scores
Version: 1.0.0
Number of assays: 1
Number of sequences: 1
Number of structures: 1
Number of MSAs: 1
Number of assay conditions: 2


## Creating a Dataset from the Manifest

Now let's create a dataset from our manifest:

In [10]:
# Create dataset from manifest
try:
    dataset = Dataset.from_manifest(manifest)
    print(f"Successfully created dataset: {dataset.name}")
    print(f"Dataset contains:")
    print(f"  - {len(dataset.assays)} assays")
    print(f"  - {len(dataset.sequences)} sequences")
    print(f"  - {len(dataset.structures)} structures")
    print(f"  - {len(dataset.msas)} MSAs")
except Exception as e:
    print(f"Error creating dataset: {e}")
    print("This might be due to missing data files. Check that all paths in the manifest exist.")

Successfully created dataset: NEIME_2019
Dataset contains:
  - 1 assays
  - 1 sequences
  - 1 structures
  - 1 MSAs


## Creating Dataset Archives

Once you have a working dataset, you can create a portable archive:

In [11]:
# Create an archive (if dataset was successfully created)
if 'dataset' in locals():
    archive_path = dataset.dump(path=Path("../example_data/"))
    print(f"Dataset archived to: {archive_path}")
    print(f"Archive size: {archive_path.stat().st_size / 1024:.1f} KB")
else:
    print("Cannot create archive - dataset creation failed")

Dataset archived to: ../example_data/NEIME_2019.zip
Archive size: 1300.7 KB


## Archive Structure

Let's examine what's inside a dataset archive:

In [12]:
import zipfile

if 'archive_path' in locals() and archive_path.exists():
    with zipfile.ZipFile(archive_path, 'r') as zip_file:
        print("Archive contents:")
        for file_info in zip_file.filelist:
            print(f"  {file_info.filename} ({file_info.file_size} bytes)")
else:
    print("Archive not available")

Archive contents:
  manifest.lock (892 bytes)
  assays/Assay1.csv (156447 bytes)
  sequences/tr|A0A1I9GEU1|A0A1I9GEU1_NEIME.fasta (245 bytes)
  structures/A0A1I9GEU1.pdb (97127 bytes)
  msas/A0A1I9GEU1_NEIME.fasta (1076529 bytes)


## Tips for Manifest Creation

### 1. File Path Organization
- Use relative paths in manifests for portability
- Organize data files in logical directories
- Keep manifest files at the root of your dataset directory

### 2. Naming Conventions
- Use descriptive names for datasets and assays
- Follow consistent naming patterns
- Include version information when appropriate

### 3. Metadata Completeness
- Always include descriptions for datasets and conditions
- Specify units for numerical conditions
- Document the source and processing of your data

### 4. Validation
- Always test your manifest by loading it
- Verify that all file paths exist and are accessible
- Check that assay conditions are properly defined



## Common Manifest Patterns

### Multiple Assays
```toml
[[assays]]
name = "binding_assay"
path = "assays/binding.csv"
target = "binding_affinity"

[[assays]]
name = "stability_assay"
path = "assays/stability.csv"
target = "melting_temp"
```

### Directory-based Data
```toml
[[structures]]
path = "structures/"  # All files in directory
```

### Complex Conditions
```toml
[[assay_conditions]]
name = "buffer_composition"
description = "Tris-HCl buffer with NaCl"
value = "50mM Tris-HCl, 150mM NaCl"
```




## Next Steps

Now that you know how to create manifests and archives, you can:

1. **Load and explore data**: See `03_Loading_and_Accessing_Data.ipynb`
2. **Create your own dataset**: Use your protein data with PG2 Dataset
3. **Share your work**: Distribute dataset archives to collaborators